In [ ]:
#以下では、学習データを用いてモデルを学習し、検証データで予測精度を評価します。
%pip install -q --no-warn-conflicts scikit-learn==1.4.2
import json
import os
import shutil
import tarfile
import time
from datetime import datetime
from io import BytesIO

import boto3
import joblib
import pandas as pd
from botocore.exceptions import ClientError
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


# ============================================================
# Settings
# ============================================================
REGION = "ap-northeast-1"
DATA_BUCKET = "techcon-initial-data-backet-751899531459"
MODEL_BUCKET = "__CFN_MODEL_ARTIFACT_BUCKET__"
TARGET = "delivery_time_min"
INSTANCE_TYPE = "ml.m5.large"
VARIANT_NAME = "AllTraffic"

FEATURE_COLS = [
    "customer_lat",
    "customer_lon",
    "delivery_fee",
    "item_count",
    "total_price",
    "menu_type_heavy",
    "is_first_order",
    "is_app_channel",
    "store_lat",
    "store_lon",
    "store_avg_prep_time",
    "is_chain_store",
]

IMAGE_URI = (
    "354813040037.dkr.ecr.ap-northeast-1.amazonaws.com/"
    "sagemaker-scikit-learn:1.4-2-py312"
)


timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

MODEL_NAME = f"delivery-time-{timestamp}"
ENDPOINT_CONFIG_NAME = f"delivery-time-config-{timestamp}"
ENDPOINT_NAME = "eats-dev-delivery-time-endpoint"

MODEL_KEY = f"models/{MODEL_NAME}/model.tar.gz"

WORK_DIR = "decision_tree_endpoint"
CODE_DIR = os.path.join(WORK_DIR, "code")
MODEL_TAR = "model.tar.gz"


# ============================================================
# AWS clients / current SageMaker execution role
# ============================================================
s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)


def get_current_execution_role():
    """Get the role attached to the current SageMaker Studio/Notebook environment."""
    try:
        from sagemaker import get_execution_role

        return get_execution_role()
    except (ImportError, AttributeError):
        from sagemaker.core.helper.session_helper import get_execution_role

        return get_execution_role()


ROLE_ARN = get_current_execution_role()
print("Execution role:", ROLE_ARN)
print("Inference image:", IMAGE_URI)


# ============================================================
# Load data from S3
# ============================================================
def read_csv_from_s3(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(
        BytesIO(response["Body"].read()),
        dtype={"store_id": "string"},
    )


orders_train = read_csv_from_s3(DATA_BUCKET, "trainingdata.csv")
store_info = read_csv_from_s3(DATA_BUCKET, "store_info.csv")

string_columns = [
    "order_id",
    "order_datetime",
    "payment_method",
    "device_type",
    "main_category",
    "store_category",
    "area_code",
]

float_columns = [
    "customer_lat",
    "customer_lon",
    "delivery_fee",
    "total_price",
    "store_lat",
    "store_lon",
]

int_columns = [
    "item_count",
    "menu_type_heavy",
    "is_first_order",
    "is_app_channel",
    "store_avg_prep_time",
    "is_chain_store",
]

def apply_dtypes_inplace(df):
    for col in string_columns:
        if col in df.columns:
            df[col] = df[col].astype("string")

    for col in float_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="raise").astype("float64")

    for col in int_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="raise").astype("Int64")


apply_dtypes_inplace(orders_train)
apply_dtypes_inplace(store_info)

print("orders_train:", orders_train.shape)
print("store_info:", store_info.shape)


# ============================================================
# Validate data
# ============================================================
if "store_id" not in store_info.columns:
    raise ValueError("store_info.csv must contain store_id")

if store_info["store_id"].isna().any():
    raise ValueError("store_info.csv contains null store_id")

if store_info["store_id"].duplicated().any():
    duplicated_ids = (
        store_info.loc[store_info["store_id"].duplicated(), "store_id"]
        .astype(str)
        .head(10)
        .tolist()
    )
    raise ValueError(
        "store_info.csv contains duplicated store_id. "
        f"examples={duplicated_ids}"
    )

# ============================================================
# Feature engineering for training
# ============================================================
training_data = orders_train.merge(
    store_info,
    on="store_id",
    how="left",
    validate="many_to_one",
)

# ============================================================
# Train / evaluate
# ============================================================
MODEL_PARAMS = {
    "random_state": 42,
}

train_data, valid_data = train_test_split(
    training_data,
    test_size=0.2,
    random_state=42,
)

model = DecisionTreeRegressor(**MODEL_PARAMS)
model.fit(
    train_data[FEATURE_COLS],
    train_data[TARGET],
)

valid_prediction = model.predict(
    valid_data[FEATURE_COLS[:-1]]
)

mae = mean_absolute_error(
    valid_data[TARGET],
    valid_prediction,
)

print(f"MAE: {mae:.3f} min")


# ============================================================
# Build model artifact
# ============================================================
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(CODE_DIR, exist_ok=True)

joblib.dump(model, os.path.join(WORK_DIR, "decision_tree_model.joblib"))

store_info.to_csv(
    os.path.join(WORK_DIR, "store_info.csv"),
    index=False,
)

with open(
    os.path.join(WORK_DIR, "feature_columns.json"),
    "w",
    encoding="utf-8",
) as file:
    json.dump(FEATURE_COLS, file, ensure_ascii=False)




In [ ]:
#以下では、SageMaker Endpoint上で実行する推論コードを記述します。
# ============================================================
# Inference code executed inside the SageMaker endpoint
# ============================================================
inference_code = r'''
import json
import os
import time
import traceback
import uuid

import joblib
import numpy as np
import pandas as pd


def _ms(start):
    return round((time.perf_counter() - start) * 1000.0, 3)


def _log(event, **fields):
    payload = {
        "event": event,
        "pid": os.getpid(),
        **fields,
    }
    print("PERF " + json.dumps(payload, ensure_ascii=False, default=str), flush=True)


def model_fn(model_dir):
    started = time.perf_counter()
    try:
        model_path = os.path.join(model_dir, "decision_tree_model.joblib")
        store_path = os.path.join(model_dir, "store_info.csv")

        model = joblib.load(model_path)

        store_info = pd.read_csv(
            store_path,
            dtype={"store_id": "string"},
        )

        if store_info["store_id"].duplicated().any():
            raise ValueError("store_info.csv contains duplicated store_id")

        with open(
            os.path.join(model_dir, "feature_columns.json"),
            encoding="utf-8",
        ) as file:
            feature_columns = json.load(file)

        bundle = {
            "model": model,
            "store_info": store_info,
            "feature_columns": feature_columns,
        }

        _log(
            "model_fn.done",
            elapsed_ms=_ms(started),
            stores=len(store_info),
            features=len(feature_columns),
            model_file_bytes=os.path.getsize(model_path),
            store_file_bytes=os.path.getsize(store_path),
        )
        return bundle
    except Exception as exc:
        _log(
            "model_fn.error",
            elapsed_ms=_ms(started),
            error=repr(exc),
        )
        traceback.print_exc()
        raise


def input_fn(request_body, content_type):
    started = time.perf_counter()
    request_id = uuid.uuid4().hex[:16]
    try:
        normalized_content_type = (
            (content_type or "")
            .split(";", 1)[0]
            .strip()
            .lower()
        )
        if normalized_content_type != "application/json":
            raise ValueError("Content-Type must be application/json")

        if isinstance(request_body, bytes):
            body_size = len(request_body)
        else:
            body_size = len(str(request_body).encode("utf-8"))

        payload = json.loads(request_body)

        if isinstance(payload, dict) and "records" in payload:
            records = payload["records"]
        elif isinstance(payload, dict):
            records = [payload]
        elif isinstance(payload, list):
            records = payload
        else:
            raise ValueError("Invalid JSON input")

        if not isinstance(records, list) or not records:
            raise ValueError("records must be a non-empty list")
        if not all(isinstance(record, dict) for record in records):
            raise ValueError("Every record must be a JSON object")

        df = pd.DataFrame.from_records(records)
        elapsed_ms = _ms(started)
        df.attrs["request_id"] = request_id
        df.attrs["request_started"] = started
        df.attrs["input_ms"] = elapsed_ms

        _log(
            "input_fn.done",
            request_id=request_id,
            elapsed_ms=elapsed_ms,
            body_bytes=body_size,
            records=len(df),
            columns=list(df.columns),
        )
        return df
    except Exception as exc:
        _log(
            "input_fn.error",
            request_id=request_id,
            elapsed_ms=_ms(started),
            error=repr(exc),
        )
        traceback.print_exc()
        raise


def create_features(orders, model_bundle, request_id):
    total_started = time.perf_counter()

    required = {"store_id"}
    missing = required - set(orders.columns)
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")

    df = orders.copy()

    # Ignore store master columns from request input and always use artifact values.
    overlapping_store_columns = [
        column
        for column in model_bundle["store_info"].columns
        if column != "store_id" and column in df.columns
    ]
    if overlapping_store_columns:
        df = df.drop(columns=overlapping_store_columns)

    merge_started = time.perf_counter()
    df["store_id"] = df["store_id"].astype("string")
    unknown_store_count = int(
        (~df["store_id"].isin(model_bundle["store_info"]["store_id"])).sum()
    )
    df = df.merge(
        model_bundle["store_info"],
        on="store_id",
        how="left",
        validate="many_to_one",
    )
    merge_ms = _ms(merge_started)

    feature_select_started = time.perf_counter()
    feature_columns = model_bundle["feature_columns"][:-1]
    df = df.reindex(columns=feature_columns)
    feature_select_ms = _ms(feature_select_started)

    timings = {
        "feature_total_ms": _ms(total_started),
        "store_merge_ms": merge_ms,
        "feature_select_ms": feature_select_ms,
    }

    _log(
        "create_features.done",
        request_id=request_id,
        rows=len(df),
        unknown_store_count=unknown_store_count,
        **timings,
    )

    return df, timings, unknown_store_count


def predict_fn(input_data, model_bundle):
    started = time.perf_counter()
    request_id = input_data.attrs.get("request_id", uuid.uuid4().hex[:16])
    request_started = input_data.attrs.get("request_started", started)
    input_ms = input_data.attrs.get("input_ms")

    try:
        if "order_id" in input_data.columns:
            order_ids = input_data["order_id"].astype(str).tolist()
        else:
            order_ids = None

        features, feature_timings, unknown_store_count = create_features(
            input_data,
            model_bundle,
            request_id,
        )

        predict_started = time.perf_counter()
        predictions = model_bundle["model"].predict(features)
        predict_ms = _ms(predict_started)
        predictions = np.maximum(predictions, 0)

        results = []
        for index, prediction in enumerate(predictions):
            result = {"delivery_time_min": float(prediction)}
            if order_ids is not None:
                result["order_id"] = order_ids[index]
            results.append(result)

        predict_fn_ms = _ms(started)
        total_through_predict_ms = round(
            (time.perf_counter() - request_started) * 1000.0,
            3,
        )

        server_timing_ms = {
            "input_fn": input_ms,
            **feature_timings,
            "model_predict": predict_ms,
            "predict_fn_total": predict_fn_ms,
            "total_through_predict": total_through_predict_ms,
        }

        _log(
            "predict_fn.done",
            request_id=request_id,
            rows=len(features),
            model_predict_ms=predict_ms,
            predict_fn_total_ms=predict_fn_ms,
            total_through_predict_ms=total_through_predict_ms,
            unknown_store_count=unknown_store_count,
        )

        return {
            "payload": {
                "request_id": request_id,
                "predictions": results,
                "server_timing_ms": server_timing_ms,
            },
            "request_id": request_id,
            "request_started": request_started,
        }
    except Exception as exc:
        _log(
            "predict_fn.error",
            request_id=request_id,
            elapsed_ms=_ms(started),
            error=repr(exc),
        )
        traceback.print_exc()
        raise


def output_fn(prediction, accept):
    started = time.perf_counter()
    request_id = prediction.get("request_id", "unknown")
    try:
        normalized_accept = (
            (accept or "application/json")
            .split(";", 1)[0]
            .strip()
            .lower()
        )
        if normalized_accept not in {"application/json", "*/*", ""}:
            raise ValueError("Accept must be application/json")

        body = json.dumps(
            prediction["payload"],
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")

        _log(
            "output_fn.done",
            request_id=request_id,
            elapsed_ms=_ms(started),
            response_bytes=len(body),
            total_request_ms=round(
                (time.perf_counter() - prediction["request_started"]) * 1000.0,
                3,
            ),
        )
        return body
    except Exception as exc:
        _log(
            "output_fn.error",
            request_id=request_id,
            elapsed_ms=_ms(started),
            error=repr(exc),
        )
        traceback.print_exc()
        raise
'''


In [ ]:
#以下では、学習済みモデルをSageMaker Endpointへデプロイし、エンドポイント経由の推論動作を検証します。
# ============================================================
# Build model artifact
# ============================================================
with open(
    os.path.join(CODE_DIR, "inference.py"),
    "w",
    encoding="utf-8",
) as file:
    file.write(inference_code)

with tarfile.open(MODEL_TAR, "w:gz") as tar:
    tar.add(
        os.path.join(WORK_DIR, "decision_tree_model.joblib"),
        arcname="decision_tree_model.joblib",
    )
    tar.add(
        os.path.join(WORK_DIR, "store_info.csv"),
        arcname="store_info.csv",
    )
    tar.add(
        os.path.join(WORK_DIR, "feature_columns.json"),
        arcname="feature_columns.json",
    )
    tar.add(CODE_DIR, arcname="code")

print("Local model artifact:", MODEL_TAR)
print("Local model artifact bytes:", os.path.getsize(MODEL_TAR))


# ============================================================
# Upload model artifact
# ============================================================
s3.upload_file(MODEL_TAR, MODEL_BUCKET, MODEL_KEY)
model_data_url = f"s3://{MODEL_BUCKET}/{MODEL_KEY}"
print("Model artifact:", model_data_url)


# ============================================================
# Create SageMaker Model
# ============================================================
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        "Image": IMAGE_URI,
        "ModelDataUrl": model_data_url,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1",
            "OMP_NUM_THREADS": "1",
            "OPENBLAS_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
            "NUMEXPR_NUM_THREADS": "1",
        },
    },
)
print("Model created:", MODEL_NAME)


# ============================================================
# Create EndpointConfig / Endpoint
# ============================================================
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": VARIANT_NAME,
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InitialVariantWeight": 1.0,
        }
    ],
)
print("EndpointConfig created:", ENDPOINT_CONFIG_NAME)

try:
    sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_exists = True
except ClientError as exc:
    if exc.response["Error"]["Code"] == "ValidationException":
        endpoint_exists = False
    else:
        raise

if endpoint_exists:
    sm.update_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=ENDPOINT_CONFIG_NAME,
    )
    print("Endpoint update started:", ENDPOINT_NAME)
else:
    sm.create_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=ENDPOINT_CONFIG_NAME,
    )
    print("Endpoint creation started:", ENDPOINT_NAME)

while True:
    detail = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = detail["EndpointStatus"]
    print("Endpoint status:", status)

    if status == "InService":
        break
    if status == "Failed":
        raise RuntimeError(
            detail.get("FailureReason", "Endpoint create/update failed")
        )
    time.sleep(20)

print("Endpoint is InService:", ENDPOINT_NAME)


# ============================================================
# Direct invocation benchmark
# IMPORTANT: This bypasses API Gateway/Lambda and measures SageMaker path.
# ============================================================
def invoke_endpoint_measured(records):
    body = json.dumps(
        {"records": records},
        ensure_ascii=False,
        default=str,
        separators=(",", ":"),
    ).encode("utf-8")

    started = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=body,
    )
    raw = response["Body"].read()
    client_ms = round((time.perf_counter() - started) * 1000.0, 3)
    result = json.loads(raw.decode("utf-8"))

    return {
        "client_ms": client_ms,
        "aws_request_id": response["ResponseMetadata"].get("RequestId"),
        "result": result,
    }


valid_records = valid_data.head(3).to_dict(orient="records")

print("\n=== Direct SageMaker invocation benchmark ===")
for call_index in range(1, 6):
    measured = invoke_endpoint_measured(valid_records)
    server_ms = measured["result"].get("server_timing_ms", {})
    print(
        f"call={call_index} "
        f"client_ms={measured['client_ms']} "
        f"server_total_ms={server_ms.get('total_through_predict')} "
        f"model_predict_ms={server_ms.get('model_predict')} "
        f"request_id={measured['result'].get('request_id')} "
        f"aws_request_id={measured['aws_request_id']}"
    )
    print(json.dumps(measured["result"], ensure_ascii=False, indent=2))
